# Multihop-easy with few-shot demonstrations — dataset builder

**Question.** Base models continue *"The food obtained from the striped flying insect that lives
in a hive is"* with a description (*"a delicacy in the Philippines"*) rather than the answer.
Can solved examples of the same question type in front of each prompt make the answer format
explicit without leaking the item's bridge or answer?

**Answer.** This notebook writes `data/evaluations/lens-eval-multihop-easy-fewshot.json`: a copy
of `lens-eval-multihop-easy.json` in which every prompt is preceded by `N_DEMOS` solved prompts of
the same relation and phrasing, chosen so that neither the item's intermediate nor its target
appears in the demonstrations. No model is used here; the effect on a model is measured in the
benchmark notebooks.

## Method

* **Demonstrations** are other items of the dataset, written as `"{prompt} {target}.\n"` and
  prepended to the item's own prompt, which stays last; the readout position (the final prompt
  token) and every annotation are unchanged.
* **Candidates**, in order of preference: same `relation` and `template_id` (phrasing); same
  `relation`; same `family`. Never the same `base_id` or `group_id` (the same bridge in another
  relation, e.g. *dog* for legs and for sound).
* **No leaks:** a candidate is dropped if its prompt or target contains the item's intermediate or
  target as a whole word (case-insensitive), or if its intermediate or target equals the item's.
  Demonstrations have pairwise different targets, so a frequent answer cannot be copied from the
  context by majority.
* **Deterministic:** within each preference tier candidates are ordered by
  `sha256(seed:item:candidate)`.
* `demo_relation_match` records whether all demonstrations share the item's relation; items whose
  relation has too few other base items (some animal relations) fall back to the family.

In [1]:
import hashlib
import json
import re
from pathlib import Path

import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "evaluations").is_dir())
SOURCE = REPO / "data/evaluations/lens-eval-multihop-easy.json"
OUTPUT = REPO / "data/evaluations/lens-eval-multihop-easy-fewshot.json"
N_DEMOS = 3
SEED = 0
source = json.loads(SOURCE.read_text())
items = source["items"]
print(len(items), "items")

200 items


In [2]:
def contains_word(text, word):
    return re.search(r"(?<!\w)" + re.escape(word) + r"(?!\w)", text, re.I) is not None


def leaks(item, candidate):
    labels = [*item["intermediates"], item["target"]]
    if set(candidate["intermediates"]) & set(item["intermediates"]) or candidate["target"] == item["target"]:
        return True
    text = f"{candidate['prompt']} {candidate['target']}"
    return any(contains_word(text, label) for label in labels)


def order(item, candidate):
    return hashlib.sha256(f"{SEED}:{item['name']}:{candidate['name']}".encode()).hexdigest()


def demos_for(item):
    tiers = [
        lambda c: c["relation"] == item["relation"] and c["template_id"] == item["template_id"],
        lambda c: c["relation"] == item["relation"],
        lambda c: c["family"] == item["family"],
    ]
    chosen, targets = [], set()
    for tier in tiers:
        pool = [c for c in items if tier(c) and c["base_id"] != item["base_id"] and c["group_id"] != item["group_id"]
                and not leaks(item, c) and c["name"] not in {d["name"] for d in chosen}]
        for candidate in sorted(pool, key=lambda c: order(item, c)):
            if len(chosen) == N_DEMOS:
                break
            if candidate["target"] in targets:
                continue
            chosen.append(candidate)
            targets.add(candidate["target"])
    if len(chosen) < N_DEMOS:
        raise ValueError(f"Not enough demonstrations for {item['name']}")
    return chosen


fewshot = []
for item in items:
    demos = demos_for(item)
    context = "".join(f"{d['prompt']} {d['target']}.\n" for d in demos)
    prompt = context + item["prompt"]
    # The dataset's guarantee still holds: no label of the item appears in its evaluation prompt.
    assert not any(contains_word(prompt, label) for label in [*item["intermediates"], item["target"]]), item["name"]
    fewshot.append({**item, "prompt": prompt, "question": item["prompt"], "demos": [d["name"] for d in demos],
                    "demo_relation_match": all(d["relation"] == item["relation"] for d in demos)})
frame = pd.DataFrame(fewshot)
display(frame.groupby(["family", "relation"]).demo_relation_match.agg(["size", "sum"]).rename(
    columns={"size": "prompts", "sum": "all demos same relation"}))

prompts  all demos same relation
family     relation                                            
animals    animal_drink              2                        0
           animal_fiber              2                        0
           animal_food               4                        0
           animal_legs              22                       22
           animal_sound              2                        0
           animal_young              8                        8
arithmetic minus_multiply           16                       14
           plus_multiply            16                       16
geography  country_continent        32                        0
           country_language         32                       32
opposites  property_antonym         40                       40
word_forms description_plural       24                       24

Three examples (the full prompt, demonstrations first):

In [3]:
for name in ["bee-food-v2", "sweden-country_continent-v1", "bottle-opposite-v1"]:
    row = frame[frame.name.eq(name)].iloc[0]
    print(f"--- {name}: intermediate {row.intermediates}, target {row.target!r}, demos {row.demos}")
    print(row.prompt, end="\n\n")

--- bee-food-v2: intermediate ['bee'], target 'honey', demos ['hen-food-v2', 'sheep-fiber-v1', 'sheep-young-v2']
The food obtained from the female farm bird that clucks is eggs.
The textile fiber obtained from the farm animal that says baa is wool.
The name for a baby of the farm animal that says baa is lamb.
The food obtained from the striped flying insect that lives in a hive is

--- sweden-country_continent-v1: intermediate ['Sweden'], target 'Europe', demos ['argentina-country_continent-v1', 'thailand-country_continent-v1', 'greece-country_language-v2']
The continent containing the country whose capital is Buenos Aires is South America.
The continent containing the country whose capital is Bangkok is Asia.
The most widely spoken language in the country with Athens as its capital is Greek.
The continent containing the country whose capital is Stockholm is

--- bottle-opposite-v1: intermediate ['empty'], target 'full', demos ['doorway-opposite-v1', 'shut-door-opposite-v1', 'mouse-siz

In [4]:
metadata = {
    **source["metadata"], "version": 1, "variant": "few-shot",
    "source": "data/evaluations/lens-eval-multihop-easy.json",
    "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "n_demos": N_DEMOS, "demo_seed": SEED,
    "demo_rule": ("same relation and template, then same relation, then same family; never the same base_id or "
                  "group_id; no demo contains the item's intermediate or target as a whole word; pairwise "
                  "different demo targets; each demo is '{prompt} {target}.' on its own line"),
    "builder": "notebooks/model_agnostic/multihop_easy/multihop_easy_fewshot_build.ipynb",
}
with OUTPUT.open("w") as f:
    f.write("{\n")
    f.write(f' "metadata": {json.dumps(metadata, ensure_ascii=False)},\n')
    f.write(' "items": [\n')
    f.write(",\n".join(f"  {json.dumps(item, ensure_ascii=False)}" for item in fewshot))
    f.write("\n ]\n}\n")
assert json.loads(OUTPUT.read_text())["items"] == fewshot
print(f"wrote {OUTPUT.relative_to(REPO)}: {len(fewshot)} items, "
      f"{int(frame.demo_relation_match.sum())} with all demonstrations of the same relation")

wrote data/evaluations/lens-eval-multihop-easy-fewshot.json: 200 items, 156 with all demonstrations of the same relation


## Conclusions

The copy keeps all 200 items, their annotations, splits and grouping fields; only `prompt` changes
(the original is kept as `question`). Load it with `load_eval(REPO, "multihop-easy-fewshot")`.
Demonstrations add other countries, animals and numbers to the context, so controls and
in-context copying should be checked when comparing lenses on this variant.